<a href="https://colab.research.google.com/github/76Falcone/AulasInteligenciaArtificial/blob/main/svm_fishmorph.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SVM – FishMorph Dataset
Pipeline: importação → análise → visualização → treino/teste + CV → Grid Search → resultados.

## 1. Importar o dataset

In [ ]:
import kagglehub
path = kagglehub.dataset_download("menegidio/fishmorph-dataset")
print(path)

In [ ]:
import os, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

sns.set_theme(style="whitegrid")

# Lista os arquivos disponíveis
arquivos = glob.glob(os.path.join(path, "**", "*.*"), recursive=True)
for a in arquivos:
    print(a)

In [ ]:
# Carrega o primeiro CSV encontrado (ajuste manualmente se houver mais de um)
csvs = [a for a in arquivos if a.lower().endswith(".csv")]
print("CSV usado:", csvs[0])
df = pd.read_csv(csvs[0])
df.head()

## 2. Análise padrão dos dados

In [ ]:
print("Shape:", df.shape)
print()
df.info()

In [ ]:
display(df.describe(include="all").T)
print("Duplicadas:", df.duplicated().sum())
print()
print("Nulos por coluna:")
nulos = df.isnull().sum()
display(nulos[nulos > 0].sort_values(ascending=False))

In [ ]:
# ---- Limpeza ----
df = df.drop_duplicates()

# remove colunas com mais de 50% de nulos
limite = 0.5 * len(df)
df = df.loc[:, df.isnull().sum() <= limite]

# remove as linhas restantes com nulos
antes = len(df)
df = df.dropna().reset_index(drop=True)
print(f"Linhas removidas por nulos: {antes - len(df)} | Shape final: {df.shape}")

## 3. Distribuição das categorias

In [ ]:
# >>> Defina aqui a coluna alvo (classe). Se None, tenta detectar automaticamente. <<<
TARGET = None

if TARGET is None:
    candidatas = [c for c in df.select_dtypes(include=["object", "category"]).columns
                  if 1 < df[c].nunique() < len(df) * 0.5]
    if not candidatas:
        raise ValueError(f"Defina TARGET manualmente. Colunas: {list(df.columns)}")
    TARGET = min(candidatas, key=lambda c: df[c].nunique())
print("Coluna alvo:", TARGET)
print("Nº de classes:", df[TARGET].nunique())

# Descarta classes muito raras (necessário para validação cruzada estratificada)
MIN_AMOSTRAS = 10
cont = df[TARGET].value_counts()
raras = cont[cont < MIN_AMOSTRAS].index
if len(raras):
    print(f"Removendo {len(raras)} classes com < {MIN_AMOSTRAS} amostras")
    df = df[~df[TARGET].isin(raras)].reset_index(drop=True)

In [ ]:
top = df[TARGET].value_counts().head(30)
plt.figure(figsize=(10, max(4, 0.3 * len(top))))
sns.barplot(x=top.values, y=top.index.astype(str), color="steelblue")
plt.title(f"Distribuição das classes – {TARGET} (top 30)")
plt.xlabel("Quantidade"); plt.ylabel("")
plt.tight_layout(); plt.show()

In [ ]:
# Visualização das features numéricas
X_num = df.select_dtypes(include=np.number)
X_num.hist(figsize=(14, 10), bins=25, color="steelblue")
plt.suptitle("Distribuição das features numéricas", y=1.02)
plt.tight_layout(); plt.show()

if X_num.shape[1] <= 25:
    plt.figure(figsize=(9, 7))
    sns.heatmap(X_num.corr(), cmap="coolwarm", center=0, annot=X_num.shape[1] <= 12, fmt=".2f")
    plt.title("Correlação entre features"); plt.show()

## 4. Divisão treino/teste + Validação cruzada

In [ ]:
X = df.select_dtypes(include=np.number).copy()
le = LabelEncoder()
y = le.fit_transform(df[TARGET])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

print("Treino:", X_train.shape, "| Teste:", X_test.shape)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

## 5. Grid Search (kernel, C e gamma – 4 valores cada)

In [ ]:
pipe = Pipeline([
    ("scaler", StandardScaler()),   # SVM é sensível à escala
    ("svm", SVC())
])

param_grid = {
    "svm__kernel": ["linear", "poly", "rbf", "sigmoid"],
    "svm__C":      [0.1, 1, 10, 100],
    "svm__gamma":  [0.001, 0.01, 0.1, 1],
}

grid = GridSearchCV(pipe, param_grid, cv=cv, scoring="accuracy",
                    n_jobs=-1, verbose=1, return_train_score=True)
grid.fit(X_train, y_train)

Fitting 5 folds for each of 64 candidates, totalling 320 fits


## 6. Melhor modelo e parâmetros

In [ ]:
print("Modelo:", grid.best_estimator_.named_steps["svm"])
print("Melhores parâmetros:", grid.best_params_)
print(f"Acurácia (média da validação cruzada): {grid.best_score_:.4f}")

res = pd.DataFrame(grid.cv_results_)
cols = ["param_svm__kernel", "param_svm__C", "param_svm__gamma", "mean_test_score", "std_test_score"]
display(res.sort_values("mean_test_score", ascending=False)[cols].head(10))

In [ ]:
# Desempenho médio por kernel
melhor_kernel = res.groupby("param_svm__kernel")["mean_test_score"].max().sort_values()
melhor_kernel.plot.barh(color="steelblue", figsize=(6, 3))
plt.xlabel("Melhor acurácia CV"); plt.title("Melhor resultado por kernel"); plt.show()

## 7. Acurácia no teste e matriz de confusão

In [ ]:
best = grid.best_estimator_
y_pred = best.predict(X_test)

print(f"Acurácia no conjunto de teste: {accuracy_score(y_test, y_pred):.4f}\n")
print(classification_report(y_test, y_pred, target_names=le.classes_.astype(str), zero_division=0))

In [ ]:
n = len(le.classes_)
cm = confusion_matrix(y_test, y_pred)

if n <= 20:
    fig, ax = plt.subplots(figsize=(max(6, n * 0.7), max(5, n * 0.6)))
    ConfusionMatrixDisplay(cm, display_labels=le.classes_).plot(ax=ax, cmap="Blues", xticks_rotation=90, colorbar=False)
else:
    # muitas classes: matriz normalizada por linha, sem anotações
    cm_n = confusion_matrix(y_test, y_pred, normalize="true")
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm_n, cmap="Blues", xticklabels=False, yticklabels=False)
    plt.xlabel("Previsto"); plt.ylabel("Real")
plt.title("Matriz de confusão – conjunto de teste")
plt.tight_layout(); plt.show()